In [ ]:
from typing import TypeVar, Generic, Callable, Tuple

# Define generic type variables for the OpenGame class
Input = TypeVar('Input')
Output = TypeVar('Output')
Payoff = TypeVar('Payoff')
Feedback = TypeVar('Feedback')

class BaseStrategy(Generic[Input, Output]):
    """
    Base class for all strategies.
    A strategy is a class that can decide an output given an input.
    It can store its own internal state (e.g., weights, memory).
    """
    def select_action(self, input_val: Input) -> Output:
        """
        Selects an action (output) based on the given input.
        This method must be implemented by concrete strategy classes.
        """
        raise NotImplementedError

Strategy = TypeVar('Strategy', bound=BaseStrategy[Input, Output])


class OpenGame(Generic[Input, Output, Payoff, Feedback, Strategy]):
    """
    A generic class representing an open game.
    """

    def play(
        self,
        strategy: Strategy,
        input_val: Input,
    ) -> Output:
        """
        Plays the game given a strategy and an input, and returns the output.

        Args:
            strategy: The strategy to use.
            input_val: The input to the game.

        Returns:
            The output of the game.
        """
        raise NotImplementedError

    def coplay(
        self,
        strategy: Strategy,
        input_val: Input,
        reward: Payoff,
    ) -> Feedback:
        """
        Co-plays the game, providing feedback based on the strategy, input, and reward.

        Args:
            strategy: The strategy to use.
            input_val: The input to the game.
            reward: The reward received.

        Returns:
            Feedback on the game play.
        """
        raise NotImplementedError

    def best_response(
        self,
        input_val: Input,
        payoff_function: Callable[[Output], Payoff],
    ) -> Callable[[Strategy, Strategy], Strategy]:
        """
        Finds the best response strategy given an input and a reward function that
        maps outputs to payoffs.  Returns a function that takes two strategies
        and returns the best response strategy.  The intent is that the two input
        strategies are the other players strategies, and the function determines
        how to best play in response to those.

        Args:
            input_val: The input to the game.
            payoff_function: A function that maps outputs to payoffs.

        Returns:
            A function that takes two strategies and returns the best response strategy.
        """
        raise NotImplementedError


Outcome: ('Cooperate', 'Cooperate')
Payoff: 3
Feedback: Strategy always_cooperate resulted in a reward of 3
Best Response Strategy against always cooperate: my_strategy
Best response outcome: ('Defect', 'Defect')
Best response payoff: 1


In [ ]:
from typing import TypeAlias

# Example Implementation: Prisoner's Dilemma

# Define type aliases for the Prisoner's Dilemma
PDAction: TypeAlias = str  # "Cooperate" or "Defect"
PDPayoff: TypeAlias = int  # Numerical representation of reward/penalty
PDFeedback: TypeAlias = str  # Can be some string indicating what happened
PDStrategy: TypeAlias = Callable[[PDAction], PDAction]  # Function to choose action
PDInput: TypeAlias = None  # No real input, can make it a turn number if needed


class PrisonerDilemma(OpenGame[
    PDInput,
    Tuple[PDAction, PDAction],
    PDPayoff,
    PDFeedback, PDStrategy]
):
    """
    An implementation of the Prisoner's Dilemma as an OpenGame.
    """

    def play(self, strategy: PDStrategy, input_val: PDInput) -> Tuple[PDAction, PDAction]:
        """
        Simulates a single round of the Prisoner's Dilemma.

        Args:
            strategy: A strategy to use (takes their own action and returns action).
            input_val: Unused in this simple implementation.

        Returns:
            A tuple containing the actions of both players (player1, player2) assuming they use the same strategy.  In a more complex setup,
            we'd take two strategies as arguments to play against each other.
        """
        action1 = strategy(None)  # Choose action based on empty input (no history)
        action2 = strategy(None)  # Same for the other player.
        return (action1, action2)

    def coplay(self, strategy: PDStrategy, input_val: PDInput, reward: PDPayoff) -> PDFeedback:
        """
        Provides feedback based on the strategy, input, and reward.

        Args:
            strategy: The strategy used.
            input_val: The input to the game.
            reward: The reward received.

        Returns:
            Feedback on the game play.
        """
        return f"Strategy {strategy.__name__} resulted in a reward of {reward}"

    def best_response(self, input_val: PDInput, reward_function: Callable[[Tuple[PDAction, PDAction]], Payoff]) -> Callable[[PDStrategy, PDStrategy], PDStrategy]:
        """
        Finds the best response strategy, given the other player's strategy.  This is a simplified example and doesn't account for repeated games.

        Args:
            input_val: The input to the game.
            reward_function: A function that maps outputs to rewards.

        Returns:
            A function that takes two strategies (other players) and returns the best response strategy.
        """

        def best_response_strategy(
            opponent_strategy: PDStrategy,
            self_strategy: PDStrategy
        ) -> PDStrategy: # added self_strategy to match signature

            def my_strategy(previous_action: PDAction = None) -> PDAction:  #  "previous_action" argument removed as it's not used
                """
                A simple strategy that defects if the opponent defects, otherwise cooperates.  This is a kind of "tit-for-tat"
                but only for one round, so not very good.
                """
                opponent_action = opponent_strategy(None)

                if opponent_action == "Defect":
                    return "Defect"
                else:
                    return "Defect"  # Always defect for the best response in a single round
            return my_strategy

        return best_response_strategy


# Example Usage
if __name__ == '__main__':
    pd = PrisonerDilemma()

    # Example Strategy: Always Cooperate
    def always_cooperate(previous_action: PDAction = None) -> PDAction:
        return "Cooperate"

    # Example Strategy: Always Defect
    def always_defect(previous_action: PDAction = None) -> PDAction:
        return "Defect"

    # Example Payoff function
    def prisoner_dilemma_payoff(actions: Tuple[PDAction, PDAction]) -> Payoff:
        """
        Defines the payoff matrix for the Prisoner's Dilemma.
        """
        action1, action2 = actions
        if action1 == "Cooperate" and action2 == "Cooperate":
            return 3  # Both cooperate
        elif action1 == "Cooperate" and action2 == "Defect":
            return 0  # Player 1 gets suckered
        elif action1 == "Defect" and action2 == "Cooperate":
            return 5  # Player 1 defects
        else:  # Both defect
            return 1

    # Play the game
    outcome = pd.play(always_cooperate, None)  # Input is None in this case
    print(f"Outcome: {outcome}")

    # Calculate payoff
    payoff = prisoner_dilemma_payoff(outcome)
    print(f"Payoff: {payoff}")

    # Get feedback
    feedback = pd.coplay(always_cooperate, None, payoff)
    print(f"Feedback: {feedback}")

    # Find the best response strategy
    best_response_func = pd.best_response(None, prisoner_dilemma_payoff)
    best_response_strategy = best_response_func(always_cooperate, always_cooperate)

    print(f"Best Response Strategy against always cooperate: {best_response_strategy.__name__}")  # the best response strategy *IS* a strategy to play against the always cooperate strategy

    best_response_outcome = pd.play(best_response_strategy, None)

    print(f"Best response outcome: {best_response_outcome}")
    print(f"Best response payoff: {prisoner_dilemma_payoff(best_response_outcome)}")